In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

import folium

## Activity Differences Check

In [ ]:
def visualize_activity_ratios(file_path):
    # Load the data
    data = pd.read_csv(file_path, names=['id', 'age', 'gender', 'occupation', 'starttime', 'duration', 'activity', 'lon', 'lat', 'citycode'], header=None, skiprows=1)
    
    # Convert 'starttime' from seconds to a readable time format (HH:MM)
    data['starttime_formatted'] = data['starttime'].apply(lambda x: (datetime.min + timedelta(seconds=x)).time())
    
    # Group the data by 'starttime_formatted' and 'activity' and count unique 'id's to consider population
    grouped_data = data.groupby(['starttime_formatted', 'activity'])['id'].nunique().unstack(fill_value=0)
    
    # Calculate the ratio of activities based on the unique 'id' counts (population)
    population_per_time = len(data['id'].unique())  # Total unique individuals per starttime
    activity_ratios = grouped_data.div(population_per_time, axis=0)[[2,3,100,200,300,400,500]]
    # print(activity_ratios[[2,3,100,200,300,400,500]])
    
    # Plotting with ordered activities and a diverse color palette
    plt.figure(figsize=(16, 6))
    activity_ratios.plot(kind='area', stacked=True, ax=plt.gca(), colormap='tab20')
    plt.title('Activity Ratios Over Time Based on Population')
    plt.xlabel('Start Time (HH:MM)')
    plt.ylabel('Activity Ratio')
    plt.xticks(rotation=45)
    plt.legend(title='Activity', loc='upper left')
    plt.tight_layout()
    
    # Show the plot
    plt.show()

# Example usage
file_path = '/mnt/large/data/PseudoPFLOW/activity/13/person_13101_labor.csv'  # Replace with your actual file path
visualize_activity_ratios(file_path)

In [ ]:
def visualize_activity_ratios(file_path):
    # Load the data
    data = pd.read_csv(file_path, names=['id', 'age', 'gender', 'occupation', 'starttime', 'duration', 'activity', 'lon', 'lat', 'citycode'], header=None, skiprows=1)
    
    # Convert 'starttime' from seconds to a readable time format (HH:MM)
    data['starttime_formatted'] = data['starttime'].apply(lambda x: (datetime.min + timedelta(seconds=x)).time())
    
    # Duplicate 00:00 data for missing time slots between 00:15 and 02:45
    midnight_data = data[data['starttime_formatted'] == datetime.strptime("00:00", "%H:%M").time()]
    for minutes in range(15, 166, 15):  # Loop from 00:15 to 02:45 in 15-minute intervals
        fill_time = (datetime.min + timedelta(minutes=minutes)).time()
        fill_data = midnight_data.copy()
        fill_data['starttime_formatted'] = fill_time
        data = pd.concat([data, fill_data])
    
    # Group the data by 'starttime_formatted' and 'activity' and count unique 'id's
    grouped_data = data.groupby(['starttime_formatted', 'activity'])['id'].nunique().unstack(fill_value=0)
    
    # Calculate the ratio of activities based on the unique 'id' counts (population)
    population_per_time = len(data['id'].unique())  # Total unique individuals per starttime
    activity_ratios = grouped_data.div(population_per_time, axis=0)[[2,3, 100,200,300,400, 500]]
    
    # Plotting with ordered activities and a diverse color palette
    plt.figure(figsize=(16, 6))
    activity_ratios.plot(kind='area', stacked=True, ax=plt.gca(), colormap='tab20')
    plt.title('Activity Ratios Over Time Based on Population')
    plt.xlabel('Start Time (HH:MM)')
    plt.ylabel('Activity Ratio')
    plt.xticks(rotation=45)
    plt.legend(title='Activity', loc='upper left')
    plt.tight_layout()
    
    # Show the plot
    plt.show()

# You can use this improved function by passing the file path as an argument.


In [ ]:
file_path = '/mnt/large/data/PseudoPFLOW/activity/1/person_01101_student.csv'
visualize_activity_ratios(file_path)

In [ ]:
def calculate_activity_ratios(file_path):
    data = pd.read_csv(file_path, names=['id', 'age', 'gender', 'occupation', 'starttime', 'duration', 'activity', 'lon', 'lat', 'citycode'], header=None, skiprows=1)
    data['starttime_formatted'] = data['starttime'].apply(lambda x: (datetime.min + timedelta(seconds=x)).time())
    
    midnight_data = data[data['starttime_formatted'] == datetime.strptime("00:00", "%H:%M").time()]
    for minutes in range(15, 166, 15):
        fill_time = (datetime.min + timedelta(minutes=minutes)).time()
        fill_data = midnight_data.copy()
        fill_data['starttime_formatted'] = fill_time
        data = pd.concat([data, fill_data])
        
    grouped_data = data.groupby(['starttime_formatted', 'activity'])['id'].nunique().unstack(fill_value=0)
    population_per_time = len(data['id'].unique())
    activity_ratios = grouped_data.div(population_per_time, axis=0)[[2,3,100,200,300,400,500]]
    
    return activity_ratios


In [ ]:
file_path1 = '/mnt/large/data/PseudoPFLOW/activity/13/person_13101_labor.csv'  # Replace with the actual path of the first file
file_path2 = '/mnt/large/data/PseudoPFLOW/activity/13_2018/person_13101_labor.csv'  # Replace with the actual path of the second file

activity_ratios_1 = calculate_activity_ratios(file_path1)
activity_ratios_2 = calculate_activity_ratios(file_path2)

# Calculate the differences
differences = activity_ratios_1 - activity_ratios_2


In [ ]:
plt.figure(figsize=(16, 6))
differences.plot(kind='line', ax=plt.gca(), colormap='tab20')  # Changed from 'area' to 'line'
plt.title('Differences in Activity Ratios Over Time')
plt.xlabel('Start Time (HH:MM)')
plt.ylabel('Difference in Activity Ratio')
plt.axhline(0, color='gray', lw=0.8)  # Adds a horizontal line at y=0 for reference
plt.xticks(rotation=45)
plt.legend(title='Activity', loc='upper left')
plt.tight_layout()
plt.show()



## Outing Ratio and Average Trips

In [ ]:
pop_root = '/Users/pang/Data/PseudoPFLOW/agent/person_12101.csv' 
act_root = '/Users/pang/Data/PseudoPFLOW/Activity1.2/12/activity_12101.csv'

pop = pd.read_csv(pop_root)
act = pd.read_csv(act_root, names=['pid', 'age', 'gender', 'occupation', 'starttime', 'duration', 'purpose', 'lon', 'lat', 'zcode'])

len(pop), len(act.groupby('pid'))

In [ ]:
root = '/Users/pang/Data/PseudoPFLOW/'
for pref in os.listdir(f'{root}Activity1.2/'):
    out, trips, pop = 0, 0, 0
    if pref == '.DS_Store':
        continue
    for filename in os.listdir(f'{root}Activity1.2/{pref}/'):
        df = pd.read_csv(f'{root}Activity1.2/{pref}/{filename}', names=['pid', 'age', 'gender', 'occupation', 'starttime', 'duration', 'purpose', 'lon', 'lat', 'zcode'])
        grouped_df = df.groupby("pid")
        df_pop = pd.read_csv(f'{root}agent/{pref}/person_{filename[9:]}')
        pop += len(df_pop)
        for pid, person in grouped_df:
            if len(person) == 1 and person['duration'].iloc[0] == 86400:
                continue
            trips += len(person)
            out += 1
    print(f'Pref: {pref} Outing ratio: {out / pop} Average Trips: {trips / pop} Pop: {pop}')

In [ ]:
root = '/mnt/large/data/PseudoPFLOW/'
for pref in os.listdir(f'{root}activity_merge2/'):
    out, trips, pop = 0, 0, 0
    if pref == '.DS_Store':
        continue
    for filename in os.listdir(f'{root}activity_merge2/{pref}/'):
        df = pd.read_csv(f'{root}activity_merge2/{pref}/{filename}', names=['pid', 'age', 'gender', 'occupation', 'starttime', 'duration', 'purpose', 'lon', 'lat', 'zcode'])
        grouped_df = df.groupby("pid")
        # df_pop = pd.read_csv(f'{root}agent/{pref}/person_{filename[9:]}')
        pop += len(grouped_df)
        for pid, person in grouped_df:
            if len(person) == 1 and person['duration'].iloc[0] == 86400:
                continue
            trips += len(person)
            out += 1
    print(f'Pref: {pref} Outing ratio: {out / pop} Average Trips: {trips / pop} Pop: {pop}')

### out, trips, pop

In [ ]:
out / pop, trips / pop

## Transport Mode

In [ ]:
walk, bike, car, train = 0, 0, 0, 0

trip = pd.read_csv('C:\\large\\PseudoPFLOW\\trip\\16\\trip_16209.csv', names=['uid', 'starttime', 'olon', 'olat', 'dlon', 'dlat', 'mode', 'purpose', 'occupation'])
trip

In [ ]:
trip['mode'].value_counts()

In [ ]:
trip = pd.read_csv('/home/mdxuser/Data/PseudoPFLOW/trip/48/trip_13101.csv', names=['uid', 'starttime', 'olon', 'olat', 'dlon', 'dlat', 'mode', 'purpose', 'occupation'])
trip

In [ ]:
trip['mode'].value_counts().values / len(trip)

In [ ]:
trip['mode'].value_counts()

In [ ]:
trip[trip['mode']!=0]['mode'].value_counts() / len(trip[trip['mode']!=0])

In [ ]:
pop = len(trip.groupby('uid'))
pop

In [ ]:
stastic = trip
stastic.values = trip['mode'].value_counts().values / pop
stastic

In [ ]:
trip['mode'].value_counts()

In [ ]:
from haversine import haversine, Unit

In [ ]:
def calculate_distance(row):
    origin = (row['olat'], row['olon'])
    destination = (row['dlat'], row['dlon'])
    return haversine(origin, destination, unit=Unit.KILOMETERS)

trip['distance'] = trip.apply(calculate_distance, axis=1)
trip

In [ ]:
trip['distance'].hist(bins=100)

In [ ]:
trip[(trip['distance']<1)&(trip['distance']>0)]#['mode'].hist()

## POI check

In [ ]:
df = pd.read_csv("/home/mdxuser/Data/PseudoPFLOW/activity_merge3/16/activity_16201.csv", names=['pid', 'age', 'gender', 'occupation', 'starttime', 'duration', 'purpose', 'lon', 'lat', 'zcode'])
df

In [ ]:
check = df[df["purpose"].isin([100, 200])]
check

In [ ]:
map_center = [df['lat'].mean(), df['lon'].mean()]

# Create a map
map = folium.Map(location=map_center, zoom_start=13)

# Add points
for index, row in check.iterrows():
    folium.Marker([row['lat'], row['lon']], popup=f'PID: {row["pid"]}').add_to(map)
    # print(index)
    if index >= 1000:
        break

# Show the map
map

# start time check

In [ ]:
trip = pd.read_csv('C:\\large\\PseudoPFLOW\\trip\\16\\trip_16209.csv', names=['uid', 'starttime', 'olon', 'olat', 'dlon', 'dlat', 'mode', 'purpose', 'occupation'])
error_ids = trip[trip['starttime']<0]['uid'].unique()

trip[trip['uid'].isin(error_ids)]

In [ ]:
act = trip = pd.read_csv('C:\\Data\\PseudoPFLOW\\activity_merge3\\16\\activity_16209.csv', names=['uid', 'age', 'gender', 'occupation', 'starttime', 'duration', 'purpose', 'olon', 'olat', 'zcode'])
act

In [ ]:
act[act['uid'].isin(error_ids)]

## ID check

In [ ]:
traj = pd.read_csv("C:/large/PseudoPFLOW/trajectory/0/trajectory_16209.csv", names=['pid', 'ut', 'timestamp', 'lon', 'lat', 'mode', 'gender', 'occupation', 'linkid'])
act = pd.read_csv("C:/Data/PseudoPFLOW/activity_merge3/0/activity_16209.csv", names=['pid', 'age', 'gender', 'occupation', 'starttime', 'duration', 'purpose', 'lon', 'lat', 'zcode'])
trip = pd.read_csv("C:/large/PseudoPFLOW/trip/0/trip_16209.csv", names=['pid', 'starttime', 'olon', 'olat', 'dlon', 'dlat', 'mode', 'purpose', 'occupation'])

In [ ]:
traj

In [ ]:
len(traj.pid.unique()), len(act.pid.unique()), len(trip.pid.unique())

In [ ]:
diff = list(set(act.pid.unique()) - set(trip.pid.unique()))

act[act["pid"].isin(diff)]

In [ ]:
len(traj.pid.unique()), len(act.pid.unique()), len(trip.pid.unique())